# Corrective RAG — Google Colab Runner

This notebook runs the **Corrective RAG** project (LangGraph + Chroma Cloud + Kimchi LLM + BGE embeddings + prompt-injection / PII guardrails) on a free Google Colab backend and exposes the Streamlit UI over a public **ngrok** tunnel.

## One-time setup in Colab

1. Open this notebook in Google Colab (the file is also at the repo root: `colab_setup.ipynb`).
2. In the left sidebar, click the **🔑 Secrets** icon and add the following six secrets:
   - `KIMCHI_API_KEY`
   - `TAVILY_API_KEY`
   - `CHROMA_API_KEY`
   - `CHROMA_TENANT`
   - `CHROMA_DATABASE`
   - `NGROK_AUTH_TOKEN`

3. Run the cells below **in order**.

The final cell will print a public `https://*.ngrok-free.app` URL — open it in any browser.

> **Note:** Colab free tier has ~12 GB RAM and no GPU. BGE-small runs on CPU, which is fine for this workload. The session resets when the runtime disconnects, so the Chroma Cloud collection is reused (it lives in the cloud, not on the VM).

In [ ]:
# Cell 1 — Install dependencies and clone the repo
!apt-get -qq install -y poppler-utils > /dev/null && echo "poppler-utils installed."
!pip install -q --upgrade pip
!pip install -q pyngrok

import os
if not os.path.exists("CORRECTIVE_RAG_FINAL"):
    !git clone https://github.com/aksri648/CORRECTIVE_RAG_FINAL.git

%cd CORRECTIVE_RAG_FINAL
%cd "corrective rag project"
!pip install -q -r requirements.txt
print("Dependencies installed.")

In [ ]:
# Cell 2 — Load API keys from Colab secrets and write them to .env
from google.colab import userdata

REQUIRED = [
    "KIMCHI_API_KEY",
    "TAVILY_API_KEY",
    "CHROMA_API_KEY",
    "CHROMA_TENANT",
    "CHROMA_DATABASE",
    "NGROK_AUTH_TOKEN",
]

missing = [k for k in REQUIRED if not userdata.get(k)]
if missing:
    raise SystemExit(
        f"Missing Colab secrets: {missing}. Add them in the left sidebar (🔑 icon) and re-run."
    )

lines = ["# Generated from Colab secrets", ""]
for key in REQUIRED:
    value = userdata.get(key)
    os.environ[key] = value
    lines.append(f"{key}={value}")

with open(".env", "w") as f:
    f.write("\n".join(lines) + "\n")
print("Wrote .env with the 6 required secrets.")

In [ ]:
# Cell 3 — Launch Streamlit with the ngrok public tunnel
# run.py prints the public URL and keeps the tunnel alive.
# Interrupt the cell (▶️ ■) to stop both Streamlit and ngrok.
!python run.py